# Day 29 of ML 30-Days Challenge

# Project - 1 (Without Tuning)

**Regression**: Predict house prices using a regression model. Apply your knowledge of preprocessing, feature engineering, and regression models to a real-world dataset. Focus on building a complete pipeline and evaluating the model's performance.

In [46]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

###1) Load data

In [47]:
df = pd.read_csv("/content/Housing.csv")

###2) Define features and target

In [48]:
target = "price"
X = df.drop(columns=[target])
y = df[target]

###3) Identify numeric and categorical columns

In [49]:
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

###4) Preprocess + simple model

In [50]:
preprocess = ColumnTransformer(
    transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_cols),
            ("cat", Pipeline(steps=[
                  ("imp", SimpleImputer(strategy="most_frequent")),
                  ("ohe", OneHotEncoder(handle_unknown="ignore")) # it will do this : simpleImpute -> OneHotEncode -> save data output to "categorical_cols"
            ]), categorical_cols),
    ],
    remainder="drop",
)

model = Pipeline(steps=[
    ("prep", preprocess),
    ("reg", LinearRegression()),
])

###5) Train/test split

In [51]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 )

###6) Fit

In [52]:
model.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['area', 'bedrooms',
                                                   'bathrooms', 'stories',
                                                   'parking']),
                                                 ('cat',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['mainroad', 'guestroom',
                                                   'basement',
                                                   'hotwaterheating',
                                                   'airconditioning',
                                                   'prefarea',
                                                   'furnishingstatus'])])),
                ('reg', LinearRegression())])

###7) Evaluate

In [53]:
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)
print(f"MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | R2: {r2:.3f}")

MAE: 970,043 | RMSE: 1,324,507 | R2: 0.653


### 8) For my own satisfaction

In [54]:
import ipywidgets as widgets
from IPython.display import display, clear_output


#simply -> numeric : slider and categorical : checkbox and categorical options : dropdown

area = widgets.IntSlider(description="Area", min=int(df["area"].min()), max=int(df["area"].max()), step=50, value=int(df["area"].median()))
bedrooms = widgets.IntSlider(description="Bedrooms", min=int(df["bedrooms"].min()), max=int(df["bedrooms"].max()), step=1, value=int(df["bedrooms"].median()))
bathrooms = widgets.IntSlider(description="Bathrooms", min=int(df["bathrooms"].min()), max=int(df["bathrooms"].max()), step=1, value=int(df["bathrooms"].median()))
stories = widgets.IntSlider(description="Stories", min=int(df["stories"].min()), max=int(df["stories"].max()), step=1, value=int(df["stories"].median()))
parking = widgets.IntSlider(description="Parking", min=int(df["parking"].min()), max=int(df["parking"].max()), step=1, value=int(df["parking"].median()))

mainroad = widgets.Checkbox(value=True, description="Main Road (yes/no)")
guestroom = widgets.Checkbox(value=False, description="Guest room (yes/no)")
basement = widgets.Checkbox(value=False, description="Basement (yes/no)")
hotwaterheating = widgets.Checkbox(value=False, description="Hot water heating (yes/no)")
airconditioning = widgets.Checkbox(value=True, description="Air conditioning (yes/no)")
prefarea = widgets.Checkbox(value=False, description="Pref Area (yes/no)")

furnishingstatus = widgets.Dropdown(
    options=["furnished", "semi-furnished", "unfurnished"],
    value="furnished",
    description="Furnishing Status"
)

predict_btn = widgets.Button(description="Predict", button_style="success")
output = widgets.Output()

def yn(val_bool):
    return "yes" if val_bool else "no"

def on_predict_clicked(b):
    with output:
        clear_output()
        # Build new row in training column order
        row = {
            "area": area.value,
            "bedrooms": bedrooms.value,
            "bathrooms": bathrooms.value,
            "stories": stories.value,
            "mainroad": yn(mainroad.value),
            "guestroom": yn(guestroom.value),
            "basement": yn(basement.value),
            "hotwaterheating": yn(hotwaterheating.value),
            "airconditioning": yn(airconditioning.value),
            "parking": parking.value,
            "prefarea": yn(prefarea.value),
            "furnishingstatus": furnishingstatus.value,
        }

        new_X = pd.DataFrame([row])
        pred_price = model.predict(new_X)
        print(f"Predicted price: ₹{pred_price[0]:.0f}")



predict_btn.on_click(on_predict_clicked) # super classic on_click() - onClick()

single= widgets.VBox([area, bedrooms, bathrooms, stories, parking, mainroad, guestroom, basement, hotwaterheating, airconditioning, prefarea, furnishingstatus])
display(single, predict_btn, output)

Button(button_style='success', description='Predict', style=ButtonStyle())

Output()